In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pyoxigraph import (
    Store,
    NamedNode,
    Triple,
    Quad,
    Literal,
    RdfFormat,
    DefaultGraph,
)
from elasticsearch import Elasticsearch

from utils import EMBEDD_MODEL_1

INDEX_NAME = os.getenv("INDEX_NAME")

/home/zbrzeznyg/miniconda3/envs/masters/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def create_kg(df, embed_model):
    EMBEDDING_SIZE = 384
    MAPPING = {
        "properties": {
            "id":   {"type": "integer"},
            "value_name": {"type": "text"},
            "value_type": {"type": "text"},
            "triplet_id": {"type": "integer"},
            "value_embedding": {
                "type": "dense_vector",
                "dims": EMBEDDING_SIZE,
                "index": True,
                "similarity": "cosine",
            }
        }
    }

    TRIPLET_MAPPING = {
        "properties": {
            "triplet_id": {"type": "integer"},
            "embedding": {
                "type": "dense_vector",
                "dims": EMBEDDING_SIZE,
                "index": True,
                "similarity": "cosine",
            }
        }
    }

    TRIPLETS_INDEX_NAME = f"{INDEX_NAME}_triplets_index"
    ENT_INDEX_NAME = f"{INDEX_NAME}_entities_index"
    es_client = Elasticsearch('http://localhost:9200')
    es_client.indices.delete(index=ENT_INDEX_NAME, ignore=[400, 404])
    es_client.indices.create(index=ENT_INDEX_NAME, mappings=MAPPING)

    es_client.indices.delete(index=TRIPLETS_INDEX_NAME, ignore=[400, 404])
    es_client.indices.create(index=TRIPLETS_INDEX_NAME, mappings=TRIPLET_MAPPING)

    BASE = "https://example.com/political-kg/"

    ENTITY = BASE + "entity/"
    PREDICATE = BASE + "predicate/"
    ASSERTION = BASE + "assertion/"
    PROPERTY = BASE + "property/"

    RDF = "http://www.w3.org/1999/02/22-rdf-syntax-ns#"
    XSD = "http://www.w3.org/2001/XMLSchema#"

    RDF_REIFIES = NamedNode(RDF + "reifies")
    XSD_DATE = NamedNode(XSD + "date")

    PROP_SPEECH_ID = NamedNode(PROPERTY + "speech_id")
    PROP_START = NamedNode(PROPERTY + "start")
    PROP_END = NamedNode(PROPERTY + "end")
    PROP_DATE = NamedNode(PROPERTY + "date")
    PROP_TRIPLET_ID = NamedNode(PROPERTY + "triplet_id")

    entity_embeddings = {}
    relation_embeddings = {}

    graph = Store()
    for _, row in tqdm(df.iterrows(), "Creating knowledge graph from triplets", total=len(df)):
        
        subject = row["subject"].replace('<', ' ').replace('>', ' ').replace('.', ' ').strip().replace("’", "").replace("(", "").replace(")", "").replace("[", "").replace("]", "").replace("\n", " ").replace("\r", " ")
        object_ = row["object"].replace('<', ' ').replace('>', ' ').replace('.', ' ').strip().replace("’", "").replace("(", "").replace(")", "").replace("[", "").replace("]", "").replace("\n", " ").replace("\r", " ")
        predicate = row["predicate"].replace('<', ' ').replace('>', ' ').replace('.', ' ').strip().replace("’", "").replace("(", "").replace(")", "").replace("[", "").replace("]", "").replace("\n", " ").replace("\r", " ")

        subject_label = subject.replace(" ", "_")
        predicate_label = predicate.replace(" ", "_")
        object_label = object_.replace(" ", "_")

        speech_id = row["speech_id"] 
        fragment_start = row["start"]
        fragment_end = row["end"]
        fragment_date = row["date"]

        try:
            subject_uri = NamedNode(ENTITY + subject_label)
            predicate_uri = NamedNode(PREDICATE + predicate_label)
            object_uri = NamedNode(ENTITY + object_label)
        except Exception as e:
            continue

        triplet_id = hash((subject, predicate, object_)) % (10 ** 8)
        statement = NamedNode(ASSERTION + str(triplet_id))

        triple_term = Triple(subject_uri, predicate_uri, object_uri)

        if subject not in entity_embeddings:
            embedding = embed_model.encode(
                subject,
                normalize_embeddings=True
            )
            entity_embeddings[subject] = embedding

        if object_ not in entity_embeddings:
            embedding = embed_model.encode(
                object_,
                normalize_embeddings=True
            )
            entity_embeddings[object_] = embedding


        if predicate not in relation_embeddings:
            embedding = embed_model.encode(
                predicate,
                normalize_embeddings=True
            )

            relation_embeddings[predicate] = embedding

        sub_embedding = entity_embeddings[subject]
        obj_embedding = entity_embeddings[object_]
        pred_embedding = relation_embeddings[predicate]

        triplet_embedding = np.average(
            [sub_embedding, obj_embedding, pred_embedding],
            axis=0,
            weights=[0.4, 0.2, 0.4]
        )
        triplet_doc = {
            "triplet_id": triplet_id,
            "embedding": triplet_embedding.tolist()
        }
        es_client.index(index=TRIPLETS_INDEX_NAME, id=triplet_doc["triplet_id"], document=triplet_doc)

        graph.add(Quad(subject_uri, predicate_uri, object_uri))
        graph.add(Quad(statement, RDF_REIFIES, triple_term))
        graph.add(Quad(statement, PROP_SPEECH_ID, Literal(speech_id)))
        graph.add(Quad(statement, PROP_START, Literal(int(fragment_start))))
        graph.add(Quad(statement, PROP_END, Literal(int(fragment_end))))
        graph.add(Quad(statement, PROP_DATE, Literal(str(fragment_date)[:10], datatype=XSD_DATE)))
        graph.add(Quad(statement, PROP_TRIPLET_ID, Literal(triplet_id)))

    for entity, embedding in tqdm(entity_embeddings.items(), "Indexing entities in Elasticsearch"):
        entity_doc = {
            "id": hash(entity) % (10 ** 8),
            "value_name": entity.replace(" ", "_"),
            "value_type": "entity",
            "value_embedding": embedding.tolist()
        }
        es_client.index(index=ENT_INDEX_NAME, id=entity_doc["id"], document=entity_doc)

    for relation, embedding in tqdm(relation_embeddings.items(), "Indexing relations in Elasticsearch"):
        relation_doc = {
            "id": hash(relation) % (10 ** 8),
            "value_name": relation.replace(" ", "_"),
            "value_type": "relation",
            "value_embedding": embedding.tolist()
        }
        es_client.index(index=ENT_INDEX_NAME, id=relation_doc["id"], document=relation_doc)
    graph.dump(
        output=f"{INDEX_NAME}.ttl",
        format=RdfFormat.TURTLE,
        from_graph=DefaultGraph(),
        prefixes={
            "rdf": RDF,
            "xsd": XSD,
            "entity": ENTITY,
            "predicate": PREDICATE,
            "assertion": ASSERTION,
            "property": PROPERTY,
        },
    )

In [7]:
new_triplets = pd.read_csv("data/new_triplets.csv")
embed_model = EMBEDD_MODEL_1

create_kg(new_triplets, embed_model)

/tmp/ipykernel_43420/1879091720.py:33: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.delete(index=ENT_INDEX_NAME, ignore=[400, 404])
/tmp/ipykernel_43420/1879091720.py:36: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  es_client.indices.delete(index=TRIPLETS_INDEX_NAME, ignore=[400, 404])
Indexing relations in Elasticsearch: 100%|██████████| 5613/5613 [00:34<00:00, 161.73it/s]
